In [1]:
import pandas as pd

In [4]:
dataset = pd.read_csv('portugal_listings.csv')

C:\Users\PC\AppData\Local\Temp\ipykernel_19508\1801581679.py:1: DtypeWarning: Columns (9,11,12,13,14,15,19) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('portugal_listings.csv')


In [5]:
print(dataset.shape)
print(dataset.head())

(174727, 23)
      Price   District                  City  \
0  780000.0  Vila Real              Valpaços   
1  228000.0       Faro  São Brás de Alportel   
2  250000.0       Faro  São Brás de Alportel   
3  250000.0       Faro  São Brás de Alportel   
4  158000.0       Faro              Portimão   

                               Town       Type EnergyCertificate  TotalArea  \
0  Carrazedo de Montenegro e Curros       Farm                NC   552450.0   
1              São Brás de Alportel  Apartment                A+      108.0   
2              São Brás de Alportel  Apartment                A+      114.0   
3              São Brás de Alportel  Apartment                A+      114.0   
4                          Portimão  Apartment                 D    21953.0   

   NumberOfBathrooms  Parking         Floor  ...  Garage Elevator  \
0                0.0      0.0           NaN  ...     NaN    False   
1                2.0      1.0  Ground Floor  ...     NaN     True   
2               

In [6]:
print(dataset.isna().sum())

Price                       419
District                      0
City                          0
Town                          2
Type                         16
EnergyCertificate            14
TotalArea                 14450
NumberOfBathrooms         15762
Parking                      32
Floor                    146798
ConstructionYear          60467
EnergyEfficiencyLevel     68247
PublishDate              107656
Garage                    68247
Elevator                     32
ElectricCarsCharging      68247
TotalRooms                88263
NumberOfBedrooms          99904
NumberOfWC               102355
ConservationStatus       145905
LivingArea                39480
LotSize                  115244
BuiltArea                104851
dtype: int64


In [ ]:
# Since the target variable "Price" has a small number of missing values, those rows will be dropped.
dataset = dataset.loc[dataset['Price'].notna()]

In [8]:
print(dataset['Price'].isna().sum())
print(dataset.shape)

0
(174308, 23)


In [ ]:
# As for the other missing values, we will consider them below.
# Since Town, Type, EnergyCertificate, Parking and Elevator each have less than 50 missing values, we will drop those rows and not much information will be lost.
dataset = dataset.loc[dataset[['Town', 'Type', 'EnergyCertificate', 'Parking', 'Elevator']].notna().all(axis=1)]

In [11]:
print(dataset.isna().sum())

Price                         0
District                      0
City                          0
Town                          0
Type                          0
EnergyCertificate             0
TotalArea                 14407
NumberOfBathrooms         15715
Parking                       0
Floor                    146428
ConstructionYear          60237
EnergyEfficiencyLevel     68072
PublishDate              107325
Garage                    68072
Elevator                      0
ElectricCarsCharging      68072
TotalRooms                88054
NumberOfBedrooms          99686
NumberOfWC               102109
ConservationStatus       145529
LivingArea                39407
LotSize                  114918
BuiltArea                104513
dtype: int64


In [ ]:
# Columns TotalArea and NumberOfBathrooms have 8-9% missing values.
# Since this is not such a low percentage, we are going to fill those values instead of dropping rows. The question is - with what, median or average (mean) value?
# We will use median, not mean, because real estate data usually contains outliers (large villas vs small apartments).

# Before filling, we are going to group rows by a related column (e.g. property Type) instead of using one global median for the whole dataset.
# Why? Because a studio apartment and a villa have very different typical areas - a single overall median would not make sense for either one.
# Grouping lets us fill each row with a median that's realistic for that specific type of property.